<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/reasoning/03_grpo_finetune_smollm_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning a model with GRPO ##

Here's where we put the theory into practice!

In [ ]:
!pip uninstall -y gcsfs gradio huggingface-hub fsspec
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off

Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: fsspec 2024.9.0
Uninstalling fsspec-2024.9.0:
  Successfully uninstalled fsspec-2024.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.42.0 requires gcsfs!=2025.5.0,!=2026.2.0,!=2026.3.0,>=2023.3.0, which is not installed.


In [ ]:
pip install --upgrade transformers trl accelerate


  Using cached huggingface_hub-1.24.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting unins

In [ ]:
!pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.19.7
    Uninstalling wandb-0.19.7:
      Successfully uninstalled wandb-0.19.7


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.45.2
    Uninstalling bitsandbytes-0.45.2:
      Successfully uninstalled bitsandbytes-0.45.2


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

We now log in to Weights & Biases for logging and monitoring the experiment, we will use it to log our fine-tuning process!

In [ ]:
import wandb

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alexandru1 (alexandru1-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

We now load the dataset, we'll use the mlabonne/smoltldr dataset, which contains a list of short stories.

In [ ]:
dataset = load_dataset('mlabonne/smoltldr')
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})

Now let's load the model

We'll use the SmolLM2-135M model.

This is a small 135M param model that runs on limited hardware, making it ideal for learning, but it's not the most powerful model out there, a larger model like SmolLM2-1.7B can also be used for fine-tuning if it meets the computing power

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype='auto',
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Now, let's load the LoRA configuration. We'll take advantage of LoRA to reduce the number of trainable parameters, and in turn the memory footprint we need to fine-tune the model.

In [ ]:
# Load LoRA
lora_config = LoraConfig(
    task_type='CAUSAL_LM',
    r=16,
    lora_alpha=32,
    target_modules='all-linear',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039


We now define the reward function:

GRPO can use any reward function to improve the model. In this case, we'll use a simple reward function that encourages the model to generate text that is 50 tokens long.

In [ ]:
# Reward function
ideal_length = 50


def reward_len(completions, **kwargs):
  return [-abs(ideal_length - len(completion)) for completion in completions]

We now define the training parameters

We'll use the GRPOConfig class to define the training arguments in a typical transformers style.

In [ ]:
# Training Arguments
training_args = GRPOConfig(
    output_dir="GRPO",
    learning_rate=2e-5,
    per_device_train_batch_size=8, # Nr. of prompts processed simultaneously per GPU during one step
    gradient_accumulation_steps=2, # Accumulates gradients over 2 steps to simulate a larger total batch size of 16 (8x2)
    #max_prompt_length=512,
    max_completion_length=96,
    num_generations=8, # GRPO specific: number of output variations generated per prompt for comparative scoring
    optim='adamw_8bit', # Uses 8-bit quantized AdamW optimizer via bitsandbytes to save up to 75% VRAM
    num_train_epochs=1,
    bf16=True, # Enables Bfloat16 mixed precision for faster training and lower memory on newer GPUs
    report_to='wandb',
    remove_unused_columns=False, # Keeps extra dataset columns intact, essential for custom GRPO reward functions
    logging_steps=5, # Logs every 5 training steps
    push_to_hub=True, # Updates model files, checkpoints, and logs in general to the Hugging Face Hub
    )

In [ ]:
# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_len,
    args=training_args,
    train_dataset=dataset['train'],
)

# Train model
wandb.init(project='GRPO')
trainer.train()

Step,Training Loss
5,0.160824
10,0.185849
15,0.135897
20,0.133687
25,0.206466
30,0.152744
35,0.173364
40,0.157780
45,0.089946
50,0.204819


Step,Training Loss
5,0.160824
10,0.185849
15,0.135897
20,0.133687
25,0.206466
30,0.152744
35,0.173364
40,0.157780
45,0.089946
50,0.204819


Now we save and publish the model to the community!

In [ ]:
merged_model = trainer.model.merge_and_unload()
merged_model.push_to_hub(
    "SmolGRPO-135M", private=False, tags=['GRPO', 'Reasoning-HuggingFace_Course']
)

Now let's generate some text with the newly fine-tuned model:

In [ ]:
prompt = """
# A long document about the Dog

The dog (Canis lupus familiaris), also referred to as the domestic dog, is a domesticated
carnivorous mammal of the family Canidae. It is part of the wolf-like canids and was the
first species to be domesticated by humans, with archaeological evidence dating back over
15,000 years. It is widely kept as a pet, working animal, and companion, but also exists
in feral states worldwide. It is uniquely valued by humans for its loyalty, companionship,
and diverse roles in hunting, herding, protection, and assistance. Its keen sense of smell
and excellent hearing make it exceptionally adapted for tracking and detection tasks. It is
a highly social pack animal with a wide range of communication methods, including vocalizations
such as barking, whining, and howling, as well as complex body language and pheromone signaling.
Dogs vary drastically in size, shape, and color due to centuries of human-directed breeding.
"""

messages = [
    {"role": "user", "content": prompt},
]


Now we can generate text with the model.

In [ ]:
# Generate text
from transformers import pipeline

generator = pipeline('text-generation', model='SmolGRPO-135M')

generate_kwargs = {
    'max_new_tokens': 256,
    'do_sample': True,
    "temperature": 0.5,
    "min_p": 0.1,
}

generated_text = generator(messages, generate_kwargs=generate_kwargs)
print(generated_text)

Compare that with the default model:

In [ ]:
# Generate text
from transformers import pipeline

generator = pipeline('text-generation', model=model)

generate_kwargs = {
    'max_new_tokens': 256,
    'do_sample': True,
    "temperature": 0.5,
    "min_p": 0.1,
}

generated_text = generator(messages, generate_kwargs=generate_kwargs)
print(generated_text)